In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import norm
from google.colab import drive

# 1. 掛載雲端硬碟
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/資料探勘/'

# --- 金融數學公式 ---
def black_scholes_call(S, K, T, r, sigma):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def find_iv_bisection(market_price, S, K, T, r):
    low, high = 1e-5, 5.0
    if market_price <= max(0, S - K * np.exp(-r * T)):
        return 0.0
    for i in range(100):
        mid = (low + high) / 2
        price = black_scholes_call(S, K, T, r, mid)
        if abs(price - market_price) < 1e-5:
            return mid
        if price > market_price:
            high = mid
        else:
            low = mid
    return mid

def smart_read_csv(file_path, is_daily=False):
    encodings = ['utf-8-sig', 'big5', 'cp950', 'gbk']
    for enc in encodings:
        try:
            if is_daily:
                # 跳過期交所檔案第二列的虛線
                df = pd.read_csv(file_path, encoding=enc, skiprows=[1], skipinitialspace=True)
            else:
                df = pd.read_csv(file_path, encoding=enc, skipinitialspace=True)
            return df
        except:
            continue
    raise ValueError(f"無法讀取檔案: {file_path}")

# --- 主程式：計算並合併回原始檔案 ---
def update_path_file():
    path_file_path = os.path.join(folder_path, 'Path_教學_0409.csv')
    df_path = smart_read_csv(path_file_path)

    # 用來存放每一天的平均 IV
    daily_avg_ivs = []
    # 用來存放當天符合條件的樣本數
    sample_counts = []

    for index, row in df_path.iterrows():
        file_name = row['File'].strip()
        daily_file_path = os.path.join(folder_path, file_name)

        if not os.path.exists(daily_file_path):
            daily_avg_ivs.append(np.nan)
            sample_counts.append(0)
            continue

        # 讀取與清洗
        df_daily = smart_read_csv(daily_file_path, is_daily=True)
        df_daily.columns = [str(c).strip() for c in df_daily.columns]

        # 篩選條件
        target_contract = str(row['Contract']).strip()
        df_daily['商品代號'] = df_daily['商品代號'].astype(str).str.strip()
        df_daily['買賣權別'] = df_daily['買賣權別'].astype(str).str.strip()
        df_daily['到期月份(週別)'] = df_daily['到期月份(週別)'].astype(str).str.strip()
        df_daily['成交數量(B or S)'] = pd.to_numeric(df_daily['成交數量(B or S)'], errors='coerce')

        filtered = df_daily[
            (df_daily['商品代號'] == 'TXO') &
            (df_daily['買賣權別'] == 'C') &
            (df_daily['到期月份(週別)'] == target_contract) &
            (df_daily['成交數量(B or S)'] > 30)
        ].copy()

        if filtered.empty:
            daily_avg_ivs.append(np.nan)
            sample_counts.append(0)
            print(f"{file_name}: 無符合條件之交易")
            continue

        # 計算該日所有符合條件合約的 IV
        S = float(row['S0'])
        r = float(row['Rf'])
        T = float(row['Maturity']) / 365.0

        ivs = []
        for _, trade in filtered.iterrows():
            K = float(trade['履約價格'])
            market_p = float(trade['成交價格'])
            ivs.append(find_iv_bisection(market_p, S, K, T, r))

        # 計算平均值
        mean_iv = np.mean(ivs)
        daily_avg_ivs.append(mean_iv)
        sample_counts.append(len(filtered))
        print(f"檔案 {file_name} 處理完成，平均 IV: {mean_iv:.4f} (樣本數: {len(filtered)})")

    # 6. 新增欄位至原始 DataFrame
    df_path['Mean_IV'] = daily_avg_ivs
    df_path['Sample_Count'] = sample_counts

    # 7. 儲存結果 (另存新檔以保安全，或覆蓋原檔)
    output_path = os.path.join(folder_path, 'Path_教學_0409_Updated.csv')
    df_path.to_csv(output_path, index=False, encoding='utf-8-sig')

    return df_path

# 執行
try:
    updated_df = update_path_file()
    print("\n--- 更新後的資料表預覽 ---")
    display(updated_df)
    print(f"\n分析結果已成功儲存至雲端硬碟：Path_教學_0409_Updated.csv")
except Exception as e:
    print(f"發生錯誤: {e}")

檔案 OptionsDaily_2020_01_02.csv 處理完成，平均 IV: 0.1357 (樣本數: 279)
檔案 OptionsDaily_2020_01_03.csv 處理完成，平均 IV: 0.1354 (樣本數: 527)
檔案 OptionsDaily_2020_01_06.csv 處理完成，平均 IV: 0.1759 (樣本數: 492)
檔案 OptionsDaily_2020_01_07.csv 處理完成，平均 IV: 0.1670 (樣本數: 1046)
檔案 OptionsDaily_2020_01_08.csv 處理完成，平均 IV: 0.1574 (樣本數: 1449)

--- 更新後的資料表預覽 ---


,Date,File,S0,Maturity,Contract,ContractExpiryDate,Rf,Mean_IV,Sample_Count
0,2020/1/2,OptionsDaily_2020_01_02.csv,12100.48,13,202001,2020/1/15,0.0109,0.135700,279
1,2020/1/3,OptionsDaily_2020_01_03.csv,12110.43,12,202001,2020/1/15,0.0109,0.135398,527
2,2020/1/6,OptionsDaily_2020_01_06.csv,11953.36,9,202001,2020/1/15,0.0109,0.175931,492
3,2020/1/7,OptionsDaily_2020_01_07.csv,11880.32,8,202001,2020/1/15,0.0109,0.166954,1046
4,2020/1/8,OptionsDaily_2020_01_08.csv,11817.10,7,202001,2020/1/15,0.0109,0.157439,1449



分析結果已成功儲存至雲端硬碟：Path_教學_0409_Updated.csv


In [8]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
from scipy.stats import norm
from tqdm import tqdm

# --- 1. 路徑與環境設定 ---
# 假設所有壓縮檔都上傳到 Colab 的 /content/ 目錄下
# 如果在雲端硬碟，請修改為 '/content/drive/MyDrive/您的資料夾/'
working_dir = '/content/'
config_file = os.path.join(working_dir, '2022加權指數_含RF_修正版.csv')
temp_extract_folder = os.path.join(working_dir, 'temp_daily_data')

# --- 2. 金融模型定義 ---

def black_scholes_call(S, K, T, r, sigma):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def find_iv_bisection(market_price, S, K, T, r):
    low, high = 1e-5, 5.0
    # 內含價值過濾：若市場價格低於理論最低價，則 IV 無意義
    if market_price <= max(0, S - K * np.exp(-r * T)):
        return 0.0
    for i in range(100):
        mid = (low + high) / 2
        price = black_scholes_call(S, K, T, r, mid)
        if abs(price - market_price) < 1e-5:
            return mid
        if price > market_price:
            high = mid
        else:
            low = mid
    return mid

# --- 3. 主分析邏輯 ---

def process_daily_zips():
    # 讀取參數檔
    try:
        df_config = pd.read_csv(config_file, encoding='utf-8-sig')
    except:
        df_config = pd.read_csv(config_file, encoding='big5')

    mean_ivs = []
    sample_counts = []

    print(f"開始處理 2022 年度資料，預計處理 {len(df_config)} 個交易日...")

    for idx, row in tqdm(df_config.iterrows(), total=len(df_config)):
        # 取得目標檔名 (例如 OptionsDaily_2022_01_03.csv)
        csv_filename = str(row['FILE']).strip()
        # 假設壓縮檔名與 CSV 名稱一致 (例如 OptionsDaily_2022_01_03.zip)
        zip_filename = csv_filename.replace('.csv', '.zip')
        zip_path = os.path.join(working_dir, zip_filename)

        # 檢查壓縮檔是否存在
        if not os.path.exists(zip_path):
            mean_ivs.append(np.nan)
            sample_counts.append(0)
            continue

        # A. 解壓縮該日檔案
        if os.path.exists(temp_extract_folder):
            shutil.rmtree(temp_extract_folder)
        os.makedirs(temp_extract_folder)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_extract_folder)

        # B. 尋找解壓後的 CSV (考量可能在子目錄)
        target_csv_path = ""
        for root, dirs, files in os.walk(temp_extract_folder):
            if csv_filename in files:
                target_csv_path = os.path.join(root, csv_filename)
                break

        if not target_csv_path:
            mean_ivs.append(np.nan)
            sample_counts.append(0)
            continue

        # C. 讀取並清洗資料
        df_daily = None
        for enc in ['big5', 'utf-8-sig', 'cp950']:
            try:
                # 期交所格式：跳過第二列虛線
                df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
                break
            except:
                continue

        if df_daily is None:
            mean_ivs.append(np.nan)
            sample_counts.append(0)
            continue

        # 欄位清理與篩選
        df_daily.columns = [str(c).strip() for c in df_daily.columns]
        df_daily['商品代號'] = df_daily['商品代號'].astype(str).str.strip()
        df_daily['買賣權別'] = df_daily['買賣權別'].astype(str).str.strip()
        df_daily['到期月份(週別)'] = df_daily['到期月份(週別)'].astype(str).str.strip()
        df_daily['成交數量(B or S)'] = pd.to_numeric(df_daily['成交數量(B or S)'], errors='coerce')

        target_contract = str(row['contract']).strip()

        filtered = df_daily[
            (df_daily['商品代號'] == 'TXO') &
            (df_daily['買賣權別'] == 'C') &
            (df_daily['到期月份(週別)'] == target_contract) &
            (df_daily['成交數量(B or S)'] > 30)
        ].copy()

        if filtered.empty:
            mean_ivs.append(np.nan)
            sample_counts.append(0)
            continue

        # D. 計算 IV
        S = float(row['收盤價(元)'])
        r = float(row['RF'])
        T = float(row['maturity']) / 365.0

        day_ivs = []
        for _, trade in filtered.iterrows():
            try:
                K = float(trade['履約價格'])
                price = float(trade['成交價格'])
                iv = find_iv_bisection(price, S, K, T, r)
                if iv > 0: day_ivs.append(iv)
            except:
                continue

        # 紀錄結果
        if day_ivs:
            mean_ivs.append(np.mean(day_ivs))
            sample_counts.append(len(day_ivs))
        else:
            mean_ivs.append(np.nan)
            sample_counts.append(0)

    # 4. 更新參數表並回傳
    df_config['Mean_IV'] = mean_ivs
    df_config['Sample_Count'] = sample_counts

    # 儲存結果
    output_path = os.path.join(working_dir, '2022_IV_Analysis_Final.csv')
    df_config.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n分析完成！結果已儲存為：{output_path}")

    # 清理暫存資料夾
    if os.path.exists(temp_extract_folder):
        shutil.rmtree(temp_extract_folder)

    return df_config

# 執行
final_results = process_daily_zips()
final_results.head()

開始處理 2022 年度資料，預計處理 246 個交易日...


  0%|          | 0/246 [00:00<?, ?it/s]/tmp/ipykernel_15863/2381384590.py:92: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
  5%|▍         | 12/246 [01:07<39:32, 10.14s/it]/tmp/ipykernel_15863/2381384590.py:92: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
  6%|▌         | 14/246 [01:12<23:29,  6.07s/it]/tmp/ipykernel_15863/2381384590.py:92: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
 10%|█         | 25/246 [02:08<30:27,  8.27s/it]/tmp/ipykernel_15863/2381384590.py:92: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set 


分析完成！結果已儲存為：/content/2022_IV_Analysis_Final.csv


,年月日,FILE,收盤價(元),contract,contractExpirydate,maturity,RF,Mean_IV,Sample_Count
0,2022/01/03,OptionsDaily_2022_01_03.csv,18270.51,202201,2022/01/19,16,0.0084,0.130656,206
1,2022/01/04,OptionsDaily_2022_01_04.csv,18526.35,202201,2022/01/19,15,0.0084,0.118167,425
2,2022/01/05,OptionsDaily_2022_01_05.csv,18499.96,202201,2022/01/19,14,0.0084,0.136826,180
3,2022/01/06,OptionsDaily_2022_01_06.csv,18367.92,202201,2022/01/19,13,0.0084,0.129164,259
4,2022/01/07,OptionsDaily_2022_01_07.csv,18169.76,202201,2022/01/19,12,0.0084,0.157780,269


In [9]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
from scipy.stats import norm
from tqdm import tqdm

# --- 1. 路徑與環境設定 ---
working_dir = '/content/'
# 使用您最新上傳的檔案作為基礎
input_file = os.path.join(working_dir, '2022_IV_Analysis_Final.csv')
temp_extract_folder = os.path.join(working_dir, 'temp_daily_analysis')

# --- 2. 金融模型定義 (增加 Put 公式) ---

def black_scholes_price(S, K, T, r, sigma, option_type='C'):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'C':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv_bisection(market_price, S, K, T, r, option_type='C'):
    low, high = 1e-5, 5.0
    # 內含價值檢查
    intrinsic_val = max(0, S - K * np.exp(-r * T)) if option_type == 'C' else max(0, K * np.exp(-r * T) - S)
    if market_price <= intrinsic_val:
        return 0.0
    for i in range(100):
        mid = (low + high) / 2
        price = black_scholes_price(S, K, T, r, mid, option_type)
        if abs(price - market_price) < 1e-5:
            return mid
        if price > market_price:
            high = mid
        else:
            low = mid
    return mid

# --- 3. 主分析邏輯 ---

def process_advanced_analysis():
    df_config = pd.read_csv(input_file, encoding='utf-8-sig')

    # 初始化新增欄位
    results = {
        'Call_IV_Mean': [], 'Call_IV_Std': [], 'Call_Count': [],
        'Put_IV_Mean': [], 'Put_IV_Std': [], 'Put_Count': [],
        'PCR_Volume': []
    }

    print("開始深化分析：計算 Put IV 與 PCR 指標...")

    for idx, row in tqdm(df_config.iterrows(), total=len(df_config)):
        csv_filename = str(row['FILE']).strip()
        zip_filename = csv_filename.replace('.csv', '.zip')
        zip_path = os.path.join(working_dir, zip_filename)

        if not os.path.exists(zip_path):
            for key in results: results[key].append(np.nan)
            continue

        # A. 解壓與讀取
        if os.path.exists(temp_extract_folder): shutil.rmtree(temp_extract_folder)
        os.makedirs(temp_extract_folder)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_extract_folder)

        target_csv_path = ""
        for root, dirs, files in os.walk(temp_extract_folder):
            if csv_filename in files:
                target_csv_path = os.path.join(root, csv_filename)
                break

        if not target_csv_path:
            for key in results: results[key].append(np.nan)
            continue

        # B. 讀取與清洗
        df_daily = None
        for enc in ['big5', 'utf-8-sig', 'cp950']:
            try:
                df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
                break
            except: continue

        if df_daily is None:
            for key in results: results[key].append(np.nan)
            continue

        df_daily.columns = [str(c).strip() for c in df_daily.columns]
        df_daily['商品代號'] = df_daily['商品代號'].astype(str).str.strip()
        df_daily['買賣權別'] = df_daily['買賣權別'].astype(str).str.strip()
        df_daily['到期月份(週別)'] = df_daily['到期月份(週別)'].astype(str).str.strip()
        df_daily['成交數量(B or S)'] = pd.to_numeric(df_daily['成交數量(B or S)'], errors='coerce').fillna(0)

        target_contract = str(row['contract']).strip()

        # 篩選基礎：TXO + 當前合約
        df_txo = df_daily[
            (df_daily['商品代號'] == 'TXO') &
            (df_daily['到期月份(週別)'] == target_contract)
        ].copy()

        # C. 計算 PCR (成交總量比)
        call_vol = df_txo[df_txo['買賣權別'] == 'C']['成交數量(B or S)'].sum()
        put_vol = df_txo[df_txo['買賣權別'] == 'P']['成交數量(B or S)'].sum()
        pcr = put_vol / call_vol if call_vol > 0 else np.nan
        results['PCR_Volume'].append(pcr)

        # D. 計算 IV (篩選成交量 > 30)
        S, r, T = float(row['收盤價(元)']), float(row['RF']), float(row['maturity']) / 365.0

        # Call 分析
        calls = df_txo[(df_txo['買賣權別'] == 'C') & (df_txo['成交數量(B or S)'] > 30)]
        c_ivs = [find_iv_bisection(float(t['成交價格']), S, float(t['履約價格']), T, r, 'C') for _, t in calls.iterrows()]
        results['Call_IV_Mean'].append(np.mean(c_ivs) if c_ivs else np.nan)
        results['Call_IV_Std'].append(np.std(c_ivs, ddof=1) if len(c_ivs) > 1 else 0.0)
        results['Call_Count'].append(len(c_ivs))

        # Put 分析
        puts = df_txo[(df_txo['買賣權別'] == 'P') & (df_txo['成交數量(B or S)'] > 30)]
        p_ivs = [find_iv_bisection(float(t['成交價格']), S, float(t['履約價格']), T, r, 'P') for _, t in puts.iterrows()]
        results['Put_IV_Mean'].append(np.mean(p_ivs) if p_ivs else np.nan)
        results['Put_IV_Std'].append(np.std(p_ivs, ddof=1) if len(p_ivs) > 1 else 0.0)
        results['Put_Count'].append(len(p_ivs))

    # 4. 更新並儲存
    for key, value in results.items():
        df_config[key] = value

    output_path = '/content/2022_Advanced_Options_Analysis.csv'
    df_config.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n分析成功！檔案已儲存至：{output_path}")
    if os.path.exists(temp_extract_folder): shutil.rmtree(temp_extract_folder)
    return df_config

# 執行
final_df = process_advanced_analysis()
display(final_df.head())

開始深化分析：計算 Put IV 與 PCR 指標...


  0%|          | 0/246 [00:00<?, ?it/s]/tmp/ipykernel_15863/2538378295.py:86: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
  5%|▍         | 12/246 [01:52<1:09:18, 17.77s/it]/tmp/ipykernel_15863/2538378295.py:86: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
  6%|▌         | 14/246 [01:58<39:40, 10.26s/it]/tmp/ipykernel_15863/2538378295.py:86: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv_path, encoding=enc, skiprows=[1], skipinitialspace=True)
 10%|█         | 25/246 [03:40<59:03, 16.03s/it]/tmp/ipykernel_15863/2538378295.py:86: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or se


分析成功！檔案已儲存至：/content/2022_Advanced_Options_Analysis.csv


,年月日,FILE,收盤價(元),contract,contractExpirydate,maturity,RF,Mean_IV,Sample_Count,Call_IV_Mean,Call_IV_Std,Call_Count,Put_IV_Mean,Put_IV_Std,Put_Count,PCR_Volume
0,2022/01/03,OptionsDaily_2022_01_03.csv,18270.51,202201,2022/01/19,16,0.0084,0.130656,206,0.130656,0.013635,206,0.181729,0.037104,210,0.964720
1,2022/01/04,OptionsDaily_2022_01_04.csv,18526.35,202201,2022/01/19,15,0.0084,0.118167,425,0.117890,0.016885,426,0.194143,0.038087,276,0.837375
2,2022/01/05,OptionsDaily_2022_01_05.csv,18499.96,202201,2022/01/19,14,0.0084,0.136826,180,0.132412,0.030061,186,0.178593,0.039179,188,0.924774
3,2022/01/06,OptionsDaily_2022_01_06.csv,18367.92,202201,2022/01/19,13,0.0084,0.129164,259,0.128174,0.021668,261,0.210090,0.046779,297,0.988419
4,2022/01/07,OptionsDaily_2022_01_07.csv,18169.76,202201,2022/01/19,12,0.0084,0.157780,269,0.157780,0.024297,269,0.194253,0.065633,217,0.885470


In [10]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
from scipy.stats import norm
from tqdm import tqdm

# --- 1. 路徑設定 ---
working_dir = '/content/'
# 使用您提供的基礎參數檔 (請確保檔名正確)
input_file = os.path.join(working_dir, '2022_Advanced_Options_Analysis.csv')
temp_extract_folder = os.path.join(working_dir, 'temp_final_run')

# --- 2. 金融模型 (BS Model & IV) ---
def black_scholes_price(S, K, T, r, sigma, option_type='C'):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'C':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv_bisection(market_price, S, K, T, r, option_type='C'):
    low, high = 1e-5, 5.0
    intrinsic_val = max(0, S - K * np.exp(-r * T)) if option_type == 'C' else max(0, K * np.exp(-r * T) - S)
    if market_price <= intrinsic_val: return 0.0
    for i in range(100):
        mid = (low + high) / 2
        price = black_scholes_price(S, K, T, r, mid, option_type)
        if abs(price - market_price) < 1e-5: return mid
        if price > market_price: high = mid
        else: low = mid
    return mid

# --- 3. 核心處理程式 ---
def run_final_integrated_analysis():
    df_config = pd.read_csv(input_file, encoding='utf-8-sig')

    # 預備存儲容器 (新增 Call_Volume 與 Put_Volume)
    final_data = {
        'Call_IV_Mean': [], 'Call_IV_Std': [], 'Call_Count': [],
        'Put_IV_Mean': [], 'Put_IV_Std': [], 'Put_Count': [],
        'Call_Volume': [], 'Put_Volume': [], 'PCR_Volume_Ratio': []
    }

    print("正在執行全年度整合分析：包含 IV、標準差及 PCR 基礎成交量...")

    for idx, row in tqdm(df_config.iterrows(), total=len(df_config)):
        csv_filename = str(row['FILE']).strip()
        zip_path = os.path.join(working_dir, csv_filename.replace('.csv', '.zip'))

        if not os.path.exists(zip_path):
            for key in final_data: final_data[key].append(np.nan)
            continue

        # 解壓與讀取
        if os.path.exists(temp_extract_folder): shutil.rmtree(temp_extract_folder)
        os.makedirs(temp_extract_folder)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_extract_folder)

        target_csv = ""
        for root, _, files in os.walk(temp_extract_folder):
            if csv_filename in files:
                target_csv = os.path.join(root, csv_filename)
                break

        if not target_csv:
            for key in final_data: final_data[key].append(np.nan)
            continue

        # 讀取交易資料
        df_daily = None
        for enc in ['big5', 'utf-8-sig', 'cp950']:
            try:
                df_daily = pd.read_csv(target_csv, encoding=enc, skiprows=[1], skipinitialspace=True)
                break
            except: continue

        if df_daily is None:
            for key in final_data: final_data[key].append(np.nan)
            continue

        # 資料清洗
        df_daily.columns = [str(c).strip() for c in df_daily.columns]
        target_contract = str(row['contract']).strip()
        df_daily['商品代號'] = df_daily['商品代號'].astype(str).str.strip()
        df_daily['買賣權別'] = df_daily['買賣權別'].astype(str).str.strip()
        df_daily['到期月份(週別)'] = df_daily['到期月份(週別)'].astype(str).str.strip()
        df_daily['成交數量(B or S)'] = pd.to_numeric(df_daily['成交數量(B or S)'], errors='coerce').fillna(0)

        # 篩選當日目標合約 (TXO)
        df_txo = df_daily[(df_daily['商品代號'] == 'TXO') & (df_daily['到期月份(週別)'] == target_contract)].copy()

        # 1. 計算 PCR 及其底層總量
        vol_c = df_txo[df_txo['買賣權別'] == 'C']['成交數量(B or S)'].sum()
        vol_p = df_txo[df_txo['買賣權別'] == 'P']['成交數量(B or S)'].sum()

        final_data['Call_Volume'].append(vol_c)
        final_data['Put_Volume'].append(vol_p)
        final_data['PCR_Volume_Ratio'].append(vol_p / vol_c if vol_c > 0 else np.nan)

        # 2. 計算 IV (篩選成交量 > 30 且 IV > 0)
        S, r, T = float(row['收盤價(元)']), float(row['RF']), float(row['maturity']) / 365.0

        def get_iv_stats(sub_df, opt_type):
            iv_list = []
            valid_trades = sub_df[sub_df['成交數量(B or S)'] > 30]
            for _, t in valid_trades.iterrows():
                iv = find_iv_bisection(float(t['成交價格']), S, float(t['履約價格']), T, r, opt_type)
                if iv > 0: iv_list.append(iv)

            if not iv_list: return np.nan, np.nan, 0
            mean_val = np.mean(iv_list)
            std_val = np.std(iv_list, ddof=1) if len(iv_list) > 1 else 0.0
            return mean_val, std_val, len(iv_list)

        # 分別計算 Call 與 Put 的 IV 統計數據
        c_mean, c_std, c_cnt = get_iv_stats(df_txo[df_txo['買賣權別'] == 'C'], 'C')
        p_mean, p_std, p_cnt = get_iv_stats(df_txo[df_txo['買賣權別'] == 'P'], 'P')

        final_data['Call_IV_Mean'].append(c_mean)
        final_data['Call_IV_Std'].append(c_std)
        final_data['Call_Count'].append(c_cnt)
        final_data['Put_IV_Mean'].append(p_mean)
        final_data['Put_IV_Std'].append(p_std)
        final_data['Put_Count'].append(p_cnt)

    # 4. 合併與清理
    # 為了避免重複欄位，先移除原本已有的分析欄位 (如果有)
    existing_cols = ['Call_IV_Mean', 'Call_IV_Std', 'Call_Count',
                     'Put_IV_Mean', 'Put_IV_Std', 'Put_Count',
                     'PCR_Volume', 'PCR_Volume_Ratio', 'Call_Volume', 'Put_Volume']
    df_config = df_config.drop(columns=[c for c in existing_cols if c in df_config.columns])

    # 寫入最新資料
    for key, value in final_data.items():
        df_config[key] = value

    # 儲存
    output_path = '/content/Final_TXO_2022_With_Volumes.csv'
    df_config.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n分析完成！最終報表已生成：{output_path}")
    if os.path.exists(temp_extract_folder): shutil.rmtree(temp_extract_folder)
    return df_config

# 執行
final_report_with_volumes = run_final_integrated_analysis()
display(final_report_with_volumes.head())

正在執行全年度整合分析：包含 IV、標準差及 PCR 基礎成交量...


  0%|          | 0/246 [00:00<?, ?it/s]/tmp/ipykernel_15863/848659597.py:78: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv, encoding=enc, skiprows=[1], skipinitialspace=True)
  5%|▍         | 12/246 [01:51<1:08:21, 17.53s/it]/tmp/ipykernel_15863/848659597.py:78: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv, encoding=enc, skiprows=[1], skipinitialspace=True)
  6%|▌         | 14/246 [01:57<38:46, 10.03s/it]/tmp/ipykernel_15863/848659597.py:78: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(target_csv, encoding=enc, skiprows=[1], skipinitialspace=True)
 10%|█         | 25/246 [03:36<56:21, 15.30s/it]/tmp/ipykernel_15863/848659597.py:78: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.


分析完成！最終報表已生成：/content/Final_TXO_2022_With_Volumes.csv


,年月日,FILE,收盤價(元),contract,contractExpirydate,maturity,RF,Mean_IV,Sample_Count,Call_IV_Mean,Call_IV_Std,Call_Count,Put_IV_Mean,Put_IV_Std,Put_Count,Call_Volume,Put_Volume,PCR_Volume_Ratio
0,2022/01/03,OptionsDaily_2022_01_03.csv,18270.51,202201,2022/01/19,16,0.0084,0.130656,206,0.130656,0.013635,206,0.181729,0.037104,210,100170,96636,0.964720
1,2022/01/04,OptionsDaily_2022_01_04.csv,18526.35,202201,2022/01/19,15,0.0084,0.118167,425,0.118167,0.015904,425,0.194143,0.038087,276,156212,130808,0.837375
2,2022/01/05,OptionsDaily_2022_01_05.csv,18499.96,202201,2022/01/19,14,0.0084,0.136826,180,0.136826,0.018075,180,0.178593,0.039179,188,102598,94880,0.924774
3,2022/01/06,OptionsDaily_2022_01_06.csv,18367.92,202201,2022/01/19,13,0.0084,0.129164,259,0.129164,0.018569,259,0.210090,0.046779,297,143860,142194,0.988419
4,2022/01/07,OptionsDaily_2022_01_07.csv,18169.76,202201,2022/01/19,12,0.0084,0.157780,269,0.157780,0.024297,269,0.200728,0.056090,210,164062,145272,0.885470
